<a href="https://colab.research.google.com/github/francianerod/SojaMonitor-AI/blob/main/01_validacao_temporal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
01 - VALIDAÇÃO TEMPORAL do modelo de classificação da estiagem da cultura da soja
SojaMonitor AI | Base: série de Dourados/MS (Embrapa Agropecuária Oeste), 1979-2023

Este script completa e estende a validação prevista na tese (RODRIGUES, 2025), Apêndice A, com as seguintes melhorias:
  1. As safras de validação agora são AVALIADAS (acurácia, precisão, recall, matriz de confusão) e não apenas contadas.
  2. O split de treino/teste passa a ser TEMPORAL, não aleatório, o que é crucial
     pois dias vizinhos compartilham 4 dos 5 dias da janela móvel, evitando vazamento
     de informações entre treino e teste de forma inadequada para séries temporais.
  3. O modelo roda em dois cenários de features (características de entrada):
     o original e um sem as variáveis derivadas do próprio rótulo (variacao_ac_5_10)
     e sem 'ano', para testar a robustez do modelo.
  4. Compara o desempenho do modelo sempre contra o baseline da classe majoritária,
     proporcionando uma referência para a performance.

Requer: pandas, numpy, scikit-learn (opcional: xgboost, imbalanced-learn)

"""
import pandas as pd                             # Importa a biblioteca pandas para manipulação de dados
import numpy as np                              # Importa a biblioteca numpy para operações numéricas
import warnings                                 # Importa a biblioteca warnings para gerenciar avisos
warnings.filterwarnings("ignore")               # Ignora avisos para uma saída mais limpa

# Importa classificadores de scikit-learn para construção de modelos
from sklearn.ensemble import (RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier)

# Importa métricas de avaliação de modelos de scikit-learn
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, roc_auc_score,
                             precision_score, recall_score, f1_score, confusion_matrix)

CAMINHO = '/content/cpao_oficial_dados_1979_2023.csv' # Define o caminho para o arquivo de dados
LIMIAR = 20.6                                         # Define o limiar de chuva para classificar como estiagem

# ---------------------------------------------------------------- dados
def carregar(caminho=CAMINHO):
    """Carrega e pré-processa os dados do arquivo CSV."""
    df = pd.read_csv(caminho, sep=';')                            # Carrega o arquivo CSV usando ';' como separador
    df['data'] = pd.to_datetime(df['data'], format='%d/%m/%Y')    # Converte a coluna 'data' para o tipo datetime
    df = df.dropna().sort_values('data').reset_index(drop=True)   # Remove linhas com valores ausentes, ordena por data e reseta o índice

    # Janela móvel ancorada na DATA (não na posição da linha)
    # Calcula a soma da chuva em janelas móveis de 5 e 10 dias
    s = df.set_index('data')['chuva'].asfreq('D')                                             # Cria uma série temporal completa com frequência diária
    df['acumulado5dias']  = s.rolling('5D',  min_periods=5).sum().reindex(df['data']).values  # Acumulado de chuva nos últimos 5 dias
    df['acumulado10dias'] = s.rolling('10D', min_periods=10).sum().reindex(df['data']).values # Acumulado de chuva nos últimos 10 dias

    df['ano']        = df['data'].dt.year                 # Extrai o ano da coluna 'data'
    df['mes']        = df['data'].dt.month                # Extrai o mês da coluna 'data'
    df['dia_do_ano'] = df['data'].dt.dayofyear            # Extrai o dia do ano da coluna 'data'

    # Calcula a variação do acumulado de 5 e 10 dias (feature derivada)
    df['variacao_ac_5_10'] = df['acumulado5dias'] / (df['acumulado10dias'] + 1) # Adiciona 1 para evitar divisão por zero

    # Cria o rótulo 'estiagem': 1 se acumulado5dias < LIMIAR, 0 caso contrário
    df['estiagem'] = (df['acumulado5dias'] < LIMIAR).astype(int)

    # Remove as linhas iniciais que podem ter NaNs devido às janelas móveis e reseta o índice
    return df.dropna(subset=['acumulado5dias', 'acumulado10dias']).reset_index(drop=True)

# --------------------------------------------------- SMOTE (fallback offline)
def balancear(X, y, k=5, seed=42):
    """Balanceia as classes usando SMOTE ou uma implementação fallback se imblearn não estiver disponível."""
    try:
        # Tenta importar SMOTE de imbalanced-learn
        from imblearn.over_sampling import SMOTE
        return SMOTE(random_state=seed).fit_resample(X, y)   # Aplica SMOTE
    except ImportError:
        # Se imblearn não estiver instalado, usa uma implementação manual de SMOTE (fallback)
        pass # Ignora o erro e continua para a implementação manual
    from sklearn.neighbors import NearestNeighbors # Importa para encontrar vizinhos mais próximos
    rng = np.random.default_rng(seed) # Gerador de números aleatórios
    X = np.asarray(X, float); y = np.asarray(y) # Converte X e y para arrays numpy
    classes, counts = np.unique(y, return_counts=True) # Conta a ocorrência de cada classe
    alvo = counts.max(); Xs, ys = [X], [y] # Define a classe majoritária como alvo de balanceamento
    for c, n in zip(classes, counts):
        if n >= alvo:
            continue # Se a classe já tem o número alvo de amostras, pula
        Xc = X[y == c] # Amostras da classe minoritária
        nn = NearestNeighbors(n_neighbors=min(k + 1, len(Xc))).fit(Xc) # Encontra vizinhos mais próximos
        _, idx = nn.kneighbors(Xc) # Índices dos vizinhos
        faltam = alvo - n # Quantas amostras faltam para balancear
        base = rng.integers(0, len(Xc), faltam) # Seleciona amostras base aleatoriamente
        viz  = idx[base, rng.integers(1, idx.shape[1], faltam)] # Seleciona vizinhos para interpoção
        lam  = rng.random((faltam, 1)) # Fatores de interpolação aleatórios
        # Gera novas amostras sintéticas
        Xs.append(Xc[base] + lam * (Xc[viz] - Xc[base]))
        ys.append(np.full(faltam, c)) # Adiciona os rótulos correspondentes às novas amostras
    return np.vstack(Xs), np.concatenate(ys) # Retorna os dados balanceados

def construir_modelos():
    """Constrói e retorna um dicionário de modelos de classificação."""
    m = {
        'RandomForest': RandomForestClassifier(random_state=42, n_estimators=200,
                                               max_depth=15, min_samples_split=5,
                                               min_samples_leaf=1), # Modelo RandomForest
        'ExtraTrees':   ExtraTreesClassifier(random_state=42, n_estimators=200,
                                             max_depth=15, min_samples_split=5,
                                             min_samples_leaf=2), # Modelo ExtraTrees
    }
    try:
        import xgboost as xgb # Tenta importar XGBoost
        m['XGBoost'] = xgb.XGBClassifier(random_state=42, eval_metric='logloss') # Adiciona XGBoost se disponível
    except ImportError:
        # Se XGBoost não estiver instalado, usa HistGradientBoostingClassifier como alternativa
        m['HistGB (~XGB)'] = HistGradientBoostingClassifier(random_state=42)
    return m # Retorna o dicionário de modelos

# ---------------------------------------------------------------- relatório
def avaliar(nome, modelo, X, y):
    """Avalia um modelo e imprime suas métricas de desempenho."""
    p  = modelo.predict(X)             # Faz previsões de classe
    pr = modelo.predict_proba(X)[:, 1] # Faz previsões de probabilidade para a classe positiva

    # Calcula a matriz de confusão e extrai TN, FP, FN, TP
    tn, fp, fn, tp = confusion_matrix(y, p, labels=[0, 1]).ravel()

    # Imprime as métricas de desempenho formatadas
    print(f"  {nome:16s} ACC {accuracy_score(y,p)*100:6.2f}%  "                            # Acurácia
          f"BAL {balanced_accuracy_score(y,p)*100:6.2f}%  AUC {roc_auc_score(y,pr):.3f}  " # Balanced Accuracy e AUC
          f"Prec {precision_score(y,p,zero_division=0):.2f}  "                             # Precisão
          f"Rec {recall_score(y,p,zero_division=0):.2f}  "                                 # Recall
          f"F1 {f1_score(y,p,zero_division=0):.2f}  |  TN{tn} FP{fp} FN{fn} TP{tp}")       # F1-score e Matriz de Confusão

def main():
    """Função principal que orquestra o carregamento, treinamento e avaliação dos modelos."""
    df = carregar() # Carrega e pré-processa os dados
    treino = df[df['data'] <= '2021-08-31'] # Define o período de treino até 31/08/2021
    safras = { # Define os períodos de validação para safras específicas
        'SAFRA RUIM 2021/22': df[(df['data'] >= '2021-09-01') & (df['data'] <= '2022-03-31')],
        'SAFRA BOA  2022/23': df[(df['data'] >= '2022-09-01') & (df['data'] <= '2023-03-31')],
    }
    cenarios = { # Define os cenários de features a serem testados
        'features originais': ['Tmedia', 'chuva', 'ano', 'mes', 'dia_do_ano', 'variacao_ac_5_10'],
        'sem vazamento':      ['Tmedia', 'chuva', 'mes', 'dia_do_ano'],
    }

    # Itera sobre cada cenário de features
    for cen, F in cenarios.items():
        print(f"\n{'='*96}\nCENARIO: {cen}  ->  {F}\n{'='*96}") # Imprime cabeçalho do cenário

        # Balanceia os dados de treino e treina os modelos para o cenário atual
        Xtr, ytr = balancear(treino[F].values, treino['estiagem'].values)
        modelos = {n: m.fit(Xtr, ytr) for n, m in construir_modelos().items()} # Treina cada modelo

        # Itera sobre cada safra para avaliação
        for nome_saf, saf in safras.items():
            y = saf['estiagem'].values # Rótulos reais da safra
            print(f"\n{nome_saf}  (n={len(saf)}, estiagem real = {y.mean()*100:.0f}% dos dias)") # Informações da safra

            # Calcula e imprime o baseline da classe majoritária
            print(f"  {'baseline maj.':16s} ACC {max(y.mean(),1-y.mean())*100:6.2f}%  BAL  50.00%  AUC 0.500")

            # Avalia cada modelo treinado na safra atual
            for nome, m in modelos.items():
                avaliar(nome, m, saf[F].values, y)

# Garante que a função main() seja chamada apenas quando o script for executado diretamente
if __name__ == '__main__':
    main()



CENARIO: features originais  ->  ['Tmedia', 'chuva', 'ano', 'mes', 'dia_do_ano', 'variacao_ac_5_10']

SAFRA RUIM 2021/22  (n=212, estiagem real = 72% dos dias)
  baseline maj.    ACC  71.70%  BAL  50.00%  AUC 0.500
  RandomForest     ACC  74.06%  BAL  71.82%  AUC 0.848  Prec 0.85  Rec 0.77  F1 0.81  |  TN40 FP20 FN35 TP117
  ExtraTrees       ACC  75.00%  BAL  78.53%  AUC 0.848  Prec 0.93  Rec 0.70  F1 0.80  |  TN52 FP8 FN45 TP107
  XGBoost          ACC  76.42%  BAL  76.49%  AUC 0.837  Prec 0.89  Rec 0.76  F1 0.82  |  TN46 FP14 FN36 TP116

SAFRA BOA  2022/23  (n=212, estiagem real = 51% dos dias)
  baseline maj.    ACC  51.42%  BAL  50.00%  AUC 0.500
  RandomForest     ACC  74.53%  BAL  74.45%  AUC 0.855  Prec 0.74  Rec 0.77  F1 0.76  |  TN74 FP29 FN25 TP84
  ExtraTrees       ACC  77.36%  BAL  77.61%  AUC 0.872  Prec 0.84  Rec 0.69  F1 0.76  |  TN89 FP14 FN34 TP75
  XGBoost          ACC  67.45%  BAL  67.49%  AUC 0.795  Prec 0.69  Rec 0.66  F1 0.68  |  TN71 FP32 FN37 TP72

CENARIO: sem 